# Extracting Cycle Representatives

This tutorial shows how to get cycle representatives out of a `PersistenceForest` object and convert them into ordinary Python and NumPy objects.

The default extraction pattern is:

```python
reps = forest.barcode_cycle_reps(relative_position=0.1, min_bar_length=0.05)
```

This returns one `SignedChain` for each selected barcode bar.


In [ ]:
import numpy as np
from persforest import PersistenceForest
from persforest.cycle_rep_vectorisations import signed_chain_to_polyhedral_paths


def sample_noisy_circle(n=120, noise=0.035, seed=4):
    rng = np.random.default_rng(seed)
    theta = np.linspace(0.0, 2.0 * np.pi, n, endpoint=False)
    theta = theta + rng.normal(scale=0.01, size=n)
    radius = 1.0 + rng.normal(scale=noise, size=n)
    return np.column_stack((radius * np.cos(theta), radius * np.sin(theta)))


def sample_noisy_sphere(n=90, noise=0.04, seed=8):
    rng = np.random.default_rng(seed)
    z = rng.uniform(-1.0, 1.0, n)
    theta = rng.uniform(0.0, 2.0 * np.pi, n)
    radius = 1.0 + rng.normal(scale=noise, size=n)
    xy = np.sqrt(1.0 - z * z)
    sphere = np.column_stack((xy * np.cos(theta), xy * np.sin(theta), z))
    return radius[:, None] * sphere


## Build a forest and extract representatives

Bars are sorted from longest to shortest internally, and `min_bar_length` filters out short bars before representatives are sampled.


In [ ]:
points = sample_noisy_circle()
forest = PersistenceForest(points)

relative_position = 0.1
min_bar_length = 0.05

reps = forest.barcode_cycle_reps(
    relative_position=relative_position,
    min_bar_length=min_bar_length,
)

len(reps)


In [ ]:
forest.plot_barcode_cycle_reps(
    relative_position=relative_position,
    min_bar_length=min_bar_length,
    coloring="bars",
    linewidth_cycle=2.0,
    style_2d={"point_color": "0.15", "point_alpha": 0.75},
)


## A `SignedChain` is a set of oriented simplices

The main field is `signed_simplices`. Each entry has the form `(simplex_tuple, orientation)`, where `orientation` is `1` or `-1`.


In [ ]:
rep = reps[0]

signed_simplices = sorted(
    (tuple(simplex), int(orientation))
    for simplex, orientation in rep.signed_simplices
)

signed_simplices[:10]


## Convert to unsigned simplices

Use `rep.unsigned()` to cancel opposite-oriented duplicate simplices. This is often the most convenient representation for downstream graph or geometry code.


In [ ]:
unsigned_simplices = rep.simplices(signed=False)

len(unsigned_simplices), unsigned_simplices[:10]


## Convert to vertex coordinates

`vertex_coordinates` returns the coordinates of all vertices touched by the chain. This forgets edge order, but it is useful for quick geometric summaries or plotting.


In [ ]:
coords = rep.vertex_coordinates(forest.point_cloud, signed=False)

coords.shape, coords[:5]


## Convert a 2D cycle into closed vertex paths

For planar 1-cycles, `signed_chain_to_polyhedral_paths` turns a signed chain into one or more closed paths. Each path is an array of vertex indices into `forest.point_cloud`.


In [ ]:
paths = signed_chain_to_polyhedral_paths(
    signed_chain=rep.unsigned(),
    point_cloud=forest.point_cloud,
)

path_vertex_indices = [path.tolist() for path in paths]
path_coordinates = [forest.point_cloud[path] for path in paths]

len(paths), path_vertex_indices[0][:10], path_coordinates[0].shape


## Extract at a specific filtration value

If you need a representative at an exact radius, get the barcode bar and call `cycle_at_filtration_value`.


In [ ]:
longest_bar = sorted(forest.barcode, key=lambda bar: bar.lifespan(), reverse=True)[0]
radius = longest_bar.birth + 0.35 * longest_bar.lifespan()
cycle_at_radius = longest_bar.cycle_at_filtration_value(radius).unsigned()

radius, len(cycle_at_radius.simplices(signed=False))


## The same extraction works in 3D

For a 3D point cloud, representatives are 2-dimensional `SignedChain` objects, so the simplices are triangles rather than edges.


In [ ]:
points_3d = sample_noisy_sphere()
forest_3d = PersistenceForest(points_3d)

surface_reps = forest_3d.barcode_cycle_reps(relative_position=0.25, min_bar_length=0.1)
surface_rep = surface_reps[0]

triangle_simplices = sorted(
    tuple(simplex)
    for simplex, _orientation in surface_rep.unsigned().signed_simplices
)

surface_rep.dim(), len(triangle_simplices), triangle_simplices[:5]
